# Document Q&A Engine — Retrieval-Augmented Generation (RAG)

This notebook builds a small end-to-end **RAG** system: given a document (plain text or PDF),
it lets you ask natural-language questions and get answers grounded in that document's content.

The pipeline has three stages:
1. **Retrieve** — find the passages most relevant to the question using vector similarity
2. **Augment** — attach those passages to the question as context
3. **Generate** — produce an answer using that context

Everything below runs fully offline by default (no model downloads needed). There's also an
optional "upgrade path" at the bottom that swaps in real sentence embeddings and a small
open-source LLM if you have internet access.

## Step 0 — Setup

We only need three lightweight, offline-friendly libraries for the default configuration:
`scikit-learn` (TF-IDF + cosine similarity), `numpy`, and `pypdf` (for reading PDF files).

In [1]:
# Lightweight, offline-friendly dependencies
%pip install -q scikit-learn numpy pypdf

# Optional: uncomment for higher-quality (but heavier) semantic search + LLM generation
# %pip install -q sentence-transformers transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 10.9 MB/s eta 0:00:00


## Step 1 — Imports

In [2]:
import os
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Step 2 — Reading the Source Document

A small helper that loads either a `.txt` file or a `.pdf` file and returns its raw text.
Swap in your own resume, lecture notes, or research paper here later.

In [3]:
def read_source_document(path: str) -> str:
    """Load a .txt or .pdf file from disk and return its plain text content."""
    file_ext = os.path.splitext(path)[1].lower()

    if file_ext == ".txt":
        with open(path, "r", encoding="utf-8") as f:
            return f.read()

    if file_ext == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(path)
        pages_text = []
        for page in reader.pages:
            pages_text.append(page.extract_text() or "")
        return "".join(pages_text)

    raise ValueError(f"Unsupported file type '{file_ext}'. Please provide a .txt or .pdf file.")

## Step 3 — Splitting the Document into Segments

Long documents need to be broken into smaller overlapping segments before we can embed them.
Smaller segments let the retriever point to the *exact* passage that answers a question,
rather than handing back the whole document. The overlap keeps sentences that straddle a
segment boundary from losing context.

In [4]:
def split_into_segments(document_text: str, segment_length: int = 150, overlap_words: int = 30):
    """Break text into overlapping word-based segments."""
    cleaned = re.sub(r"\s+", " ", document_text).strip()
    tokens = cleaned.split(" ")

    segments = []
    cursor = 0
    while cursor < len(tokens):
        end = cursor + segment_length
        piece = " ".join(tokens[cursor:end])
        if piece.strip():
            segments.append(piece)
        cursor += segment_length - overlap_words  # step forward, re-using the overlap

    return segments

## Step 4 — Turning Segments into Vectors

`VectorEncoder` converts text into numeric vectors so we can compare similarity mathematically.

- `mode="tfidf"` (default) — purely statistical, zero downloads, works well when the
  question shares keywords with the document.
- `mode="semantic"` — uses a real sentence-embedding model so it can match meaning
  (e.g. "car" ↔ "automobile") instead of exact words. Needs `sentence-transformers`.

In [5]:
class VectorEncoder:
    def __init__(self, mode: str = "tfidf"):
        self.mode = mode
        if mode == "semantic":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
        else:
            self.vectorizer = TfidfVectorizer(stop_words="english")
            self.is_fitted = False

    def fit(self, corpus):
        if self.mode == "tfidf":
            self.vectorizer.fit(corpus)
            self.is_fitted = True

    def encode(self, texts):
        if self.mode == "semantic":
            return np.array(self.model.encode(texts))

        if not self.is_fitted:
            raise RuntimeError("Call .fit(corpus) before .encode() when using tfidf mode.")
        return self.vectorizer.transform(texts).toarray()

## Step 5 — A Minimal Similarity Index

`SimilarityIndex` is a tiny in-memory vector store. It just keeps the segment vectors around
and ranks them by cosine similarity to a query. For a real multi-document system you'd
replace this with **FAISS**, **Chroma**, or **Pinecone** — but for a single document, plain
NumPy is more than enough.

In [6]:
class SimilarityIndex:
    """In-memory store that ranks text segments by cosine similarity to a query vector."""

    def __init__(self):
        self.vectors = None
        self.segments = []

    def load(self, segments, vectors):
        self.segments = segments
        self.vectors = vectors

    def top_matches(self, query_vector, k=3):
        similarity_scores = cosine_similarity(query_vector, self.vectors)[0]
        best_indices = np.argsort(similarity_scores)[::-1][:k]
        return [(self.segments[i], float(similarity_scores[i])) for i in best_indices]

## Step 6 — Producing the Final Answer

`ResponseSynthesizer` turns the retrieved segments plus the original question into an answer.

- `mode="extractive"` (default) — hands back the single highest-scoring segment verbatim.
  No downloads, always works.
- `mode="flan-t5"` — feeds the question + context into `google/flan-t5-base` for a more
  natural, abstractive answer. Needs `transformers` + `torch`.

In [7]:
class ResponseSynthesizer:
    def __init__(self, mode: str = "extractive"):
        self.mode = mode
        if mode == "flan-t5":
            from transformers import pipeline
            self.llm = pipeline("text2text-generation", model="google/flan-t5-base")

    def synthesize(self, question: str, matches) -> str:
        context_block = "\n\n".join(segment for segment, _ in matches)

        if self.mode == "flan-t5":
            prompt = (
                "Answer the question using only the context below. "
                "If the answer isn't in the context, say you don't know.\n\n"
                f"Context:\n{context_block}\n\nQuestion: {question}\nAnswer:"
            )
            output = self.llm(prompt, max_length=200)
            return output[0]["generated_text"]

        # Extractive fallback: just return the best-matching segment
        best_segment, _ = matches[0]
        return best_segment

## Step 7 — Wiring It All Together

`DocQAEngine` chains everything: read the file → split into segments → encode → index →
retrieve → synthesize an answer.

In [8]:
class DocQAEngine:
    def __init__(self, encoder_mode="tfidf", synthesizer_mode="extractive",
                 segment_length=150, overlap_words=30):
        self.encoder = VectorEncoder(mode=encoder_mode)
        self.synthesizer = ResponseSynthesizer(mode=synthesizer_mode)
        self.index = SimilarityIndex()
        self.segment_length = segment_length
        self.overlap_words = overlap_words

    def ingest(self, file_path: str):
        """Read a document and build the searchable index for it."""
        document_text = read_source_document(file_path)
        segments = split_into_segments(document_text, self.segment_length, self.overlap_words)
        self.encoder.fit(segments)
        vectors = self.encoder.encode(segments)
        self.index.load(segments, vectors)
        print(f"Indexed {len(segments)} segments from '{file_path}'")

    def ask(self, question: str, k: int = 3, show_sources: bool = True):
        """Ask a question and get back an answer plus the supporting segments."""
        question_vector = self.encoder.encode([question])
        matches = self.index.top_matches(question_vector, k=k)
        answer = self.synthesizer.synthesize(question, matches)

        if show_sources:
            print(f"\nQ: {question}")
            print(f"A: {answer}\n")
            print("Supporting segments:")
            for rank, (segment, score) in enumerate(matches, 1):
                print(f"  [{rank}] (score={score:.3f}) {segment[:120]}...")

        return answer, matches

## Step 8 — A Sample Document to Try It On

So this notebook runs immediately without any external file, the cell below writes a short
sample document to disk. **Replace this step with your own `.txt` or `.pdf` file** (notes,
resume, research paper, etc.) to actually use the engine on something real — that's the whole
point of RAG.

In [9]:
sample_notes = """Retrieval-Augmented Generation, or RAG, is a technique that pairs a
language model with an external knowledge source instead of relying only on what the model
memorized during training. When a user asks a question, the system first searches a
collection of documents for the passages most likely to contain the answer, then passes
those passages to the language model alongside the question. The model uses that supplied
context to write its response.

The main advantage of this approach is that answers stay grounded in real, checkable source
material, which reduces the chance of the model inventing facts. It also means the knowledge
base can be updated at any time simply by adding or editing documents, with no need to
retrain the underlying model.

Before documents can be searched, they are usually split into smaller overlapping segments.
Splitting matters because it lets the retrieval step point to the specific passage that
answers a question, rather than returning an entire lengthy document. Each segment is then
converted into a numerical vector, and a similarity search (commonly cosine similarity)
ranks segments by how closely they match the question.

RAG systems are widely used for chatbots that answer questions about internal company
documents, customer support tools that search product manuals, legal and medical assistants
that cite specific source passages, and personal tools that let someone ask questions about
their own notes, resumes, or research papers."""

with open("sample_notes.txt", "w", encoding="utf-8") as f:
    f.write(sample_notes)

## Step 9 — Build the Index

In [10]:
engine = DocQAEngine(encoder_mode="tfidf", synthesizer_mode="extractive")
engine.ingest("sample_notes.txt")

Indexed 2 segments from 'sample_notes.txt'


## Step 10 — Ask Some Questions

In [11]:
sample_questions = [
    "What problem does RAG solve?",
    "Why are documents split into segments before indexing?",
    "Where is RAG commonly used?",
]

for question in sample_questions:
    engine.ask(question)


Q: What problem does RAG solve?
A: the underlying model. Before documents can be searched, they are usually split into smaller overlapping segments. Splitting matters because it lets the retrieval step point to the specific passage that answers a question, rather than returning an entire lengthy document. Each segment is then converted into a numerical vector, and a similarity search (commonly cosine similarity) ranks segments by how closely they match the question. RAG systems are widely used for chatbots that answer questions about internal company documents, customer support tools that search product manuals, legal and medical assistants that cite specific source passages, and personal tools that let someone ask questions about their own notes, resumes, or research papers.

Supporting segments:
  [1] (score=0.086) the underlying model. Before documents can be searched, they are usually split into smaller overlapping segments. Splitt...
  [2] (score=0.075) Retrieval-Augmented Genera

## Step 11 — Using Your Own Document

```python
engine = DocQAEngine(encoder_mode="tfidf", synthesizer_mode="extractive")
engine.ingest("your_resume.pdf")   # or notes.txt, research_paper.pdf, etc.
engine.ask("What projects has this person worked on?")
```

## Step 12 — Upgrading to Semantic Search + a Real LLM

If you have internet access (e.g. on Colab), swap the modes for higher-quality results:

```python
# pip install sentence-transformers transformers torch

engine = DocQAEngine(encoder_mode="semantic", synthesizer_mode="flan-t5")
engine.ingest("your_document.pdf")
engine.ask("What is the main idea of this document?")
```
